In [1]:
!nvidia-smi
!nvidia-smi --query-gpu=name,compute_cap --format=csv

Fri Oct  9 06:55:31 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   49C    P8             11W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

# Exercise 01

In [2]:
%%writefile ex1_timed.cu
#include <cstdio>
#include <cstdlib>
#include <sys/time.h>
#define N 512
__global__ void add(int *a, int *b, int *c) {
 // each block runs only one addition
 c[blockIdx.x] = a[blockIdx.x] + b[blockIdx.x];
}
void random_ints(int *x, int size) {
 for (int i = 0; i < size; i++)
 x[i] = rand() % 100;
}
// Track CPU (wall-clock) time, in seconds
double cpuSecond() {
 struct timeval tp;
 gettimeofday(&tp, NULL);
 return (double)tp.tv_sec + (double)tp.tv_usec * 1.e-6;
}
int main(void) {
 int *a, *b, *c; // host copies of a, b, c
 int *d_a, *d_b, *d_c; // device copies of a, b, c
 int size = N * sizeof(int);
 double dStart, dElaps;
 // Allocate space for device copies of a, b, c
 cudaMalloc((void **)&d_a, size);
 cudaMalloc((void **)&d_b, size);
 cudaMalloc((void **)&d_c, size);
 // Allocate space for host copies of a, b, c and set up input values
 a = (int *)malloc(size); random_ints(a, N);
 b = (int *)malloc(size); random_ints(b, N);
 c = (int *)malloc(size);
 // Copy inputs to device
 cudaMemcpy(d_a, a, size, cudaMemcpyHostToDevice);
 cudaMemcpy(d_b, b, size, cudaMemcpyHostToDevice);
 // initialize timer
 dStart = cpuSecond();
 // Launch add() kernel on GPU with N blocks
 add<<<N, 1>>>(d_a, d_b, d_c);
 cudaDeviceSynchronize(); // wait for the GPU to finish, THEN stop the clock
 dElaps = cpuSecond() - dStart;
 // Copy result back to host
 cudaMemcpy(c, d_c, size, cudaMemcpyDeviceToHost);
 for (int r = 0; r < 5; r++)
 printf("%d + %d = %d\n", a[r], b[r], c[r]);
 // Check the GPU's answers against the CPU
 int errors = 0;
 for (int r = 0; r < N; r++)
 if (c[r] != a[r] + b[r]) errors++;
 printf("Verification: %d mismatches out of %d\n", errors, N);
 printf("Time taken - %.7f\n", dElaps);
 // Cleanup
 free(a); free(b); free(c);
 cudaFree(d_a); cudaFree(d_b); cudaFree(d_c);
 return 0;
}

Writing ex1_timed.cu


In [3]:
!nvcc -arch=sm_75 ex1_timed.cu -o ex1_timed && ./ex1_timed

83 + 77 = 160
86 + 34 = 120
77 + 90 = 167
15 + 26 = 41
93 + 24 = 117
Verification: 0 mismatches out of 512
Time taken - 0.0004981


### Part b

Lab 09 - Exercise 03

In [32]:
%%writefile ex3_blocks.cu
#include <cstdio>
#include <cstdlib>
#include <sys/time.h>
#define N 512
__global__ void add(int *a, int *b, int *c) {
 // each block runs only one addition
 c[blockIdx.x] = a[blockIdx.x] + b[blockIdx.x];
}
void random_ints(int *x, int size) {
 for (int i = 0; i < size; i++)
 x[i] = rand() % 100;
}
double cpuSecond() {
 struct timeval tp;
 gettimeofday(&tp, NULL);
 return (double)tp.tv_sec + (double)tp.tv_usec * 1.e-6;
}
int main(void) {
 int *a, *b, *c; // host copies of a, b, c
 int *d_a, *d_b, *d_c; // device copies of a, b, c
 int size = N * sizeof(int);
 // Allocate space for device copies of a, b, c
 cudaMalloc((void **)&d_a, size);
 cudaMalloc((void **)&d_b, size);
 cudaMalloc((void **)&d_c, size);
 // Allocate space for host copies of a, b, c and set up input values
 a = (int *)malloc(size); random_ints(a, N);
 b = (int *)malloc(size); random_ints(b, N);
 c = (int *)malloc(size);
 // Copy inputs to device
 cudaMemcpy(d_a, a, size, cudaMemcpyHostToDevice);
 cudaMemcpy(d_b, b, size, cudaMemcpyHostToDevice);
 // Launch add() kernel on GPU with N blocks, 1 thread per block
 double dStart = cpuSecond();
 add<<<N, 1>>>(d_a, d_b, d_c);
 cudaError_t syncErr = cudaDeviceSynchronize();
 double dElaps = cpuSecond() - dStart;
 if (syncErr != cudaSuccess)
    printf("Kernel execution failed: %s\n",
           cudaGetErrorString(syncErr));
 printf("Time taken - %.7f seconds\n", dElaps);
 cudaError_t err = cudaGetLastError();
 if (err != cudaSuccess)
 printf("Kernel launch failed: %s\n", cudaGetErrorString(err));
 // Copy result back to host
 cudaMemcpy(c, d_c, size, cudaMemcpyDeviceToHost);
 // Check the GPU's answers against the CPU
  int errors = 0;
  for (int r = 0; r < N; r++)
  if (c[r] != a[r] + b[r]) errors++;
  printf("Verification: %d mismatches out of %d\n", errors, N);
  // Cleanup
 free(a); free(b); free(c);
 cudaFree(d_a); cudaFree(d_b); cudaFree(d_c);
 return 0;
}

Overwriting ex3_blocks.cu


In [33]:
!nvcc -arch=sm_75 ex3_blocks.cu -o ex3_blocks && ./ex3_blocks

Time taken - 0.0001168 seconds
Verification: 0 mismatches out of 512


Lab 09 - Exercise 04

In [34]:
%%writefile ex4_threads.cu
#include <cstdio>
#include <cstdlib>
#include <sys/time.h>
#define N 512
__global__ void addT(int *a, int *b, int *c) {
 c[threadIdx.x] = a[threadIdx.x] + b[threadIdx.x];
}
void random_ints(int *x, int size) {
 for (int i = 0; i < size; i++)
 x[i] = rand() % 100;
}
double cpuSecond() {
 struct timeval tp;
 gettimeofday(&tp, NULL);
 return (double)tp.tv_sec + (double)tp.tv_usec * 1.e-6;
}
int main(void) {
 int *a, *b, *c; // host copies of a, b, c
 int *d_a, *d_b, *d_c; // device copies of a, b, c
 int size = N * sizeof(int);
 // Allocate space for device copies of a, b, c
 cudaMalloc((void **)&d_a, size);
 cudaMalloc((void **)&d_b, size);
 cudaMalloc((void **)&d_c, size);
 // Allocate space for host copies of a, b, c and set up input values
 a = (int *)malloc(size); random_ints(a, N);
 b = (int *)malloc(size); random_ints(b, N);
 c = (int *)malloc(size);
 // Copy inputs to device
 cudaMemcpy(d_a, a, size, cudaMemcpyHostToDevice);
 cudaMemcpy(d_b, b, size, cudaMemcpyHostToDevice);
 // Launch addT() kernel on GPU with 1 block of N threads
 double dStart = cpuSecond();
 addT<<<1, N>>>(d_a, d_b, d_c);
 cudaError_t syncErr = cudaDeviceSynchronize();
 double dElaps = cpuSecond() - dStart;
 if (syncErr != cudaSuccess)
    printf("Kernel execution failed: %s\n",
           cudaGetErrorString(syncErr));
 printf("Time taken - %.7f seconds\n", dElaps);
 cudaError_t err = cudaGetLastError();
 if (err != cudaSuccess)
 printf("Kernel launch failed: %s\n", cudaGetErrorString(err));
 // Copy result back to host
 cudaMemcpy(c, d_c, size, cudaMemcpyDeviceToHost);
 // Check the GPU's answers against the CPU
 int errors = 0;
 for (int r = 0; r < N; r++)
 if (c[r] != a[r] + b[r]) errors++;
 printf("Verification: %d mismatches out of %d\n", errors, N);
 // Cleanup
 free(a); free(b); free(c);
 cudaFree(d_a); cudaFree(d_b); cudaFree(d_c);
 return 0;
}

Overwriting ex4_threads.cu


In [35]:
!nvcc -arch=sm_75 ex4_threads.cu -o ex4_threads && ./ex4_threads

Time taken - 0.0000949 seconds
Verification: 0 mismatches out of 512


Lab 09 - Exercise 05

In [36]:
%%writefile ex5_vecmul.cu
#include <cstdio>
#include <cstdlib>
#include <sys/time.h>

#define N 10000000
#define THREADS_PER_BLOCK 512

__global__ void vecMul(int *a, int *b, int *c, int n) {
    // TODO 1: Global index of this thread
    int i = blockIdx.x * blockDim.x + threadIdx.x;

    // TODO 2: Make sure the thread is inside the array
    if (i < n) {
        c[i] = a[i] * b[i];
    }
}

void random_ints(int *x, int size) {
    for (int i = 0; i < size; i++) {
        x[i] = rand() % 100;
    }
}

double cpuSecond() { // 2. above main()
 struct timeval tp;
 gettimeofday(&tp, NULL);
 return (double)tp.tv_sec + (double)tp.tv_usec * 1.e-6;
}

int main(void) {
    int *a, *b, *c;
    int *d_a, *d_b, *d_c;

    size_t size = (size_t)N * sizeof(int);

    // Allocate device memory
    cudaMalloc((void **)&d_a, size);
    cudaMalloc((void **)&d_b, size);
    cudaMalloc((void **)&d_c, size);

    // Allocate host memory and generate input values
    a = (int *)malloc(size);
    random_ints(a, N);

    b = (int *)malloc(size);
    random_ints(b, N);

    c = (int *)malloc(size);

    // Copy inputs from CPU to GPU
    cudaMemcpy(d_a, a, size, cudaMemcpyHostToDevice);
    cudaMemcpy(d_b, b, size, cudaMemcpyHostToDevice);

    // TODO 3: Calculate number of blocks, rounding up
    int blocks = (N + THREADS_PER_BLOCK - 1) / THREADS_PER_BLOCK;


    // TODO 4: Launch kernel
    double dStart = cpuSecond();

    vecMul<<<blocks, THREADS_PER_BLOCK>>>(d_a, d_b, d_c, N);

    cudaError_t syncErr = cudaDeviceSynchronize();

    double dElaps = cpuSecond() - dStart;

    if (syncErr != cudaSuccess)
        printf("Kernel execution failed: %s\n",
              cudaGetErrorString(syncErr));

    printf("Time taken - %.7f seconds\n", dElaps);

    cudaError_t err = cudaGetLastError();

    if (err != cudaSuccess) {
        printf("Kernel launch failed: %s\n",
               cudaGetErrorString(err));
    }

    // TODO 5: Copy result from GPU back to CPU
    cudaMemcpy(c, d_c, size, cudaMemcpyDeviceToHost);

    // TODO 7: Verify all results
    int errors = 0;

    for (int i = 0; i < N; i++) {
        if (c[i] != a[i] * b[i]) {
            errors++;
        }
    }

    printf("Verification: %d mismatches out of %d\n",
           errors, N);

    // Cleanup
    free(a);
    free(b);
    free(c);

    cudaFree(d_a);
    cudaFree(d_b);
    cudaFree(d_c);

    return 0;
}

Overwriting ex5_vecmul.cu


In [37]:
!nvcc -arch=sm_75 ex5_vecmul.cu -o ex5_vecmul && ./ex5_vecmul

Time taken - 0.0006039 seconds
Verification: 0 mismatches out of 10000000


Lab 09 - Exercise 06

In [42]:
%%writefile ex6_matrix.cu
#include <cstdio>
#include <cstdlib>
#include <sys/time.h>

#define ROWS 10000
#define COLS 10000

// Element-wise product:
// C[r][c] = A[r][c] * B[r][c]
__global__ void mulElem(int *a, int *b, int *c, int rows, int cols) {

    // TODO 1: Calculate row and column for this thread
    int row = blockIdx.y * blockDim.y + threadIdx.y;
    int col = blockIdx.x * blockDim.x + threadIdx.x;

    // TODO 2: Check BOTH dimensions are inside the matrix
    if (row < rows && col < cols) {

        // TODO 3: Convert (row, col) to flat array index
        int idx = row * cols + col;

        c[idx] = a[idx] * b[idx];
    }
}

void random_ints(int *x, size_t size) {
    for (size_t i = 0; i < size; i++) {
        x[i] = rand() % 100;
    }
}

double cpuSecond() {
 struct timeval tp;
 gettimeofday(&tp, NULL);
 return (double)tp.tv_sec + (double)tp.tv_usec * 1.e-6;
}

int main(void) {

    int *a, *b, *c;
    int *d_a, *d_b, *d_c;

    size_t count = (size_t)ROWS * COLS;
    size_t size = count * sizeof(int);

    // Allocate GPU memory
    cudaMalloc((void **)&d_a, size);
    cudaMalloc((void **)&d_b, size);
    cudaMalloc((void **)&d_c, size);

    // Allocate CPU memory
    a = (int *)malloc(size);
    random_ints(a, count);

    b = (int *)malloc(size);
    random_ints(b, count);

    c = (int *)malloc(size);

    // Copy A and B from CPU to GPU
    cudaMemcpy(d_a, a, size, cudaMemcpyHostToDevice);
    cudaMemcpy(d_b, b, size, cudaMemcpyHostToDevice);

    // 32 x 16 = 512 threads per block
    dim3 threadsPerBlock(32, 16);

    // TODO 4: Calculate blocks in x and y, rounding UP
    dim3 numBlocks(
        (COLS + threadsPerBlock.x - 1) / threadsPerBlock.x,
        (ROWS + threadsPerBlock.y - 1) / threadsPerBlock.y
    );

    // TODO 5: Launch kernel
  double dStart = cpuSecond();

  mulElem<<<numBlocks, threadsPerBlock>>>(
      d_a, d_b, d_c, ROWS, COLS
  );

  cudaError_t syncErr = cudaDeviceSynchronize();

  double dElaps = cpuSecond() - dStart;

  if (syncErr != cudaSuccess)
      printf("Kernel execution failed: %s\n",
            cudaGetErrorString(syncErr));

    printf("Time taken - %.7f seconds\n", dElaps);

    cudaError_t err = cudaGetLastError();

    if (err != cudaSuccess) {
        printf("Kernel launch failed: %s\n",
               cudaGetErrorString(err));
    }

    // TODO 6: Copy result from GPU back to CPU
    cudaMemcpy(c, d_c, size, cudaMemcpyDeviceToHost);

    // TODO 8: Verify all results on CPU
    size_t errors = 0;

    for (size_t i = 0; i < count; i++) {
        if (c[i] != a[i] * b[i]) {
            errors++;
        }
    }

    printf("Verification: %zu mismatches out of %zu\n",
           errors, count);

    // Cleanup
    free(a);
    free(b);
    free(c);

    cudaFree(d_a);
    cudaFree(d_b);
    cudaFree(d_c);

    return 0;
}

Overwriting ex6_matrix.cu


In [43]:
!nvcc -arch=sm_75 ex6_matrix.cu -o ex6_matrix && ./ex6_matrix

Time taken - 0.0050640 seconds
Verification: 0 mismatches out of 100000000


### Part c

In [44]:
%%writefile ex5_vecmul.cu
#include <cstdio>
#include <cstdlib>
#include <sys/time.h>

#define N 100000000
#define THREADS_PER_BLOCK 512

__global__ void vecMul(int *a, int *b, int *c, int n) {
    // TODO 1: Global index of this thread
    int i = blockIdx.x * blockDim.x + threadIdx.x;

    // TODO 2: Make sure the thread is inside the array
    if (i < n) {
        c[i] = a[i] * b[i];
    }
}

void random_ints(int *x, int size) {
    for (int i = 0; i < size; i++) {
        x[i] = rand() % 100;
    }
}

double cpuSecond() { // 2. above main()
 struct timeval tp;
 gettimeofday(&tp, NULL);
 return (double)tp.tv_sec + (double)tp.tv_usec * 1.e-6;
}

int main(void) {
    int *a, *b, *c;
    int *d_a, *d_b, *d_c;

    size_t size = (size_t)N * sizeof(int);

    // Allocate device memory
    cudaMalloc((void **)&d_a, size);
    cudaMalloc((void **)&d_b, size);
    cudaMalloc((void **)&d_c, size);

    // Allocate host memory and generate input values
    a = (int *)malloc(size);
    random_ints(a, N);

    b = (int *)malloc(size);
    random_ints(b, N);

    c = (int *)malloc(size);

    // Copy inputs from CPU to GPU
    cudaMemcpy(d_a, a, size, cudaMemcpyHostToDevice);
    cudaMemcpy(d_b, b, size, cudaMemcpyHostToDevice);

    // TODO 3: Calculate number of blocks, rounding up
    int blocks = (N + THREADS_PER_BLOCK - 1) / THREADS_PER_BLOCK;


    // TODO 4: Launch kernel
    double dStart = cpuSecond();

    vecMul<<<blocks, THREADS_PER_BLOCK>>>(d_a, d_b, d_c, N);

    cudaError_t syncErr = cudaDeviceSynchronize();

    double dElaps = cpuSecond() - dStart;

    if (syncErr != cudaSuccess)
        printf("Kernel execution failed: %s\n",
              cudaGetErrorString(syncErr));

    printf("Time taken - %.7f seconds\n", dElaps);

    cudaError_t err = cudaGetLastError();

    if (err != cudaSuccess) {
        printf("Kernel launch failed: %s\n",
               cudaGetErrorString(err));
    }

    // TODO 5: Copy result from GPU back to CPU
    cudaMemcpy(c, d_c, size, cudaMemcpyDeviceToHost);

    // TODO 7: Verify all results
    int errors = 0;

    for (int i = 0; i < N; i++) {
        if (c[i] != a[i] * b[i]) {
            errors++;
        }
    }

    printf("Verification: %d mismatches out of %d\n",
           errors, N);

    // Cleanup
    free(a);
    free(b);
    free(c);

    cudaFree(d_a);
    cudaFree(d_b);
    cudaFree(d_c);

    return 0;
}

Overwriting ex5_vecmul.cu


In [45]:
!nvcc -arch=sm_75 ex5_vecmul.cu -o ex5_vecmul && ./ex5_vecmul

Time taken - 0.0048099 seconds
Verification: 0 mismatches out of 100000000


# Exercise 02

In [46]:
%%writefile ex2_check.cu
#include <cstdio>
#include <cstdlib>
#include <sys/time.h>

#define N 512

// Check CUDA runtime calls for errors
#define CHECK(call)                                                   \
    do {                                                              \
        const cudaError_t error = (call);                            \
        if (error != cudaSuccess) {                                  \
            printf("Error: %s:%d, ", __FILE__, __LINE__);            \
            printf("code:%d, reason: %s\n", error,                   \
                   cudaGetErrorString(error));                       \
            exit(1);                                                 \
        }                                                             \
    } while (0)

__global__ void add(int *a, int *b, int *c) {
    c[blockIdx.x] = a[blockIdx.x] + b[blockIdx.x];
}

void random_ints(int *x, int size) {
    for (int i = 0; i < size; i++)
        x[i] = rand() % 100;
}

double cpuSecond() {
    struct timeval tp;
    gettimeofday(&tp, NULL);
    return (double)tp.tv_sec + (double)tp.tv_usec * 1.e-6;
}

int main(void) {
    int *a, *b, *c;
    int *d_a, *d_b, *d_c;
    int size = N * sizeof(int);
    double dStart, dElaps;

    // TODO 1: Wrap all cudaMalloc calls
    CHECK(cudaMalloc((void **)&d_a, size));
    CHECK(cudaMalloc((void **)&d_b, size));
    CHECK(cudaMalloc((void **)&d_c, size));

    a = (int *)malloc(size);
    random_ints(a, N);

    b = (int *)malloc(size);
    random_ints(b, N);

    c = (int *)malloc(size);

    // TODO 2: Wrap host-to-device copies
    CHECK(cudaMemcpy(d_a, a, size, cudaMemcpyHostToDevice));
    CHECK(cudaMemcpy(d_b, b, size, cudaMemcpyHostToDevice));

    dStart = cpuSecond();

    add<<<N, 1>>>(d_a, d_b, d_c);

    // TODO 3: Check launch and synchronization
    CHECK(cudaGetLastError());
    CHECK(cudaDeviceSynchronize());

    dElaps = cpuSecond() - dStart;

    // TODO 4: Wrap device-to-host copy
    CHECK(cudaMemcpy(c, d_c, size, cudaMemcpyDeviceToHost));

    for (int r = 0; r < 5; r++)
        printf("%d + %d = %d\n", a[r], b[r], c[r]);

    int errors = 0;

    for (int r = 0; r < N; r++)
        if (c[r] != a[r] + b[r])
            errors++;

    printf("Verification: %d mismatches out of %d\n", errors, N);
    printf("Time taken - %.7f\n", dElaps);

    free(a);
    free(b);
    free(c);

    // TODO 5: Wrap cudaFree calls
    CHECK(cudaFree(d_a));
    CHECK(cudaFree(d_b));
    CHECK(cudaFree(d_c));

    return 0;
}


Writing ex2_check.cu


In [47]:
!nvcc -arch=sm_75 ex2_check.cu -o ex2_check && ./ex2_check

83 + 77 = 160
86 + 34 = 120
77 + 90 = 167
15 + 26 = 41
93 + 24 = 117
Verification: 0 mismatches out of 512
Time taken - 0.0001011


# Exercise 03

In [48]:

%%writefile ex3_picture.cu
#include <cstdio>
#include <cstdlib>
#include <sys/time.h>

#define WIDTH 800
#define HEIGHT 600

#define CHECK(call)                                                   \
    do {                                                              \
        const cudaError_t error = (call);                            \
        if (error != cudaSuccess) {                                  \
            printf("Error: %s:%d, ", __FILE__, __LINE__);            \
            printf("code:%d, reason: %s\n", error,                   \
                   cudaGetErrorString(error));                       \
            exit(1);                                                 \
        }                                                             \
    } while (0)

double cpuSecond() {
    struct timeval tp;
    gettimeofday(&tp, NULL);
    return (double)tp.tv_sec + (double)tp.tv_usec * 1.e-6;
}

// Each GPU thread processes one pixel
__global__ void PictureKernel(unsigned char *d_Pin,
                              unsigned char *d_Pout,
                              int height, int width) {

    int Row = blockIdx.y * blockDim.y + threadIdx.y;
    int Col = blockIdx.x * blockDim.x + threadIdx.x;

    if ((Row < height) && (Col < width)) {
        int v = 2 * d_Pin[Row * width + Col];
        d_Pout[Row * width + Col] = (v > 255) ? 255 : v;
    }
}

int main(void) {
    size_t count = (size_t)WIDTH * HEIGHT;
    size_t size = count * sizeof(unsigned char);

    unsigned char *h_in, *h_out;
    unsigned char *d_in, *d_out;

    double dStart, dElaps;

    h_in = (unsigned char *)malloc(size);
    h_out = (unsigned char *)malloc(size);

    CHECK(cudaMalloc((void **)&d_in, size));
    CHECK(cudaMalloc((void **)&d_out, size));

    // TODO 1: Generate random pixel values from 0 to 255
    for (size_t i = 0; i < count; i++) {
        h_in[i] = rand() % 256;
    }

    CHECK(cudaMemcpy(d_in, h_in, size, cudaMemcpyHostToDevice));

    // TODO 2: 16 x 16 threads per block
    dim3 threadsPerBlock(16, 16);

    // TODO 3: Calculate grid dimensions, rounding up
    dim3 numBlocks(
        (WIDTH + threadsPerBlock.x - 1) / threadsPerBlock.x,
        (HEIGHT + threadsPerBlock.y - 1) / threadsPerBlock.y
    );

    printf("Grid: %u x %u blocks of %u x %u threads\n",
           numBlocks.x, numBlocks.y,
           threadsPerBlock.x, threadsPerBlock.y);

    dStart = cpuSecond();

    // TODO 4: Launch the CUDA kernel
    PictureKernel<<<numBlocks, threadsPerBlock>>>(
        d_in, d_out, HEIGHT, WIDTH
    );

    CHECK(cudaGetLastError());
    CHECK(cudaDeviceSynchronize());

    dElaps = cpuSecond() - dStart;

    CHECK(cudaMemcpy(h_out, d_out, size, cudaMemcpyDeviceToHost));

    // TODO 5: Print the top-left 50 x 50 pixels
    for (int row = 0; row < 50; row++) {
        for (int col = 0; col < 50; col++) {
            int idx = row * WIDTH + col;

            printf("%3d ", (int)h_out[idx]);
        }
        printf("\n");
    }

    // TODO 6: Verify every pixel using the CPU
    size_t errors = 0;

    for (size_t i = 0; i < count; i++) {
        int expected = 2 * h_in[i];

        if (expected > 255) {
            expected = 255;
        }

        if (h_out[i] != expected) {
            errors++;
        }
    }

    printf("Verification: %zu mismatches out of %zu\n",
           errors, count);

    printf("Time taken - %.7f seconds\n", dElaps);

    free(h_in);
    free(h_out);

    CHECK(cudaFree(d_in));
    CHECK(cudaFree(d_out));

    return 0;
}

Writing ex3_picture.cu


In [49]:
!nvcc -arch=sm_75 ex3_picture.cu -o ex3_picture && ./ex3_picture

Grid: 50 x 38 blocks of 16 x 16 threads
206 255 210 230 162 255 148 255  82 255 255 255 255 255 255 140 248 255 168 255  54 255 255 255 236 180  92 198 102 255 255 255 204 100  26 255  98 176 255 180  74 186  10  46 176 255 188 255 255 255 
192  68  20 255 154  98 255 182   6 255  26  68  26 142 186 255 255 255 240 172 255 224 152 255 255 255  30 255 255 255 255 166  26 255 255 180 255 255 255 255 160 255 255 186  18 180  84 255 255 255 
255 255 255 142  32 255 255 255 255 118  72 162  62  58 214 255 180 250  32 255  46 255 255 122 255  72 255 255 255 255 156 255 255  84 255 255 255 238 255 200 255 255 255 255 255  64 255 106 255 255 
255 255 255 255 255 255 255 255 255 232 108 255  48 255  68 255 255   8  68 255 218 255 255 255 255 255 255 255 255 230 255  68 142 172 255 255   4 255 255 255 130 255  56 180 255 126 118 182 134 186 
255 255 255   4 255 255 204 184  54 255 255 255 255  84 255 255 206 255 248 255 255 218  52 118 234 255 158 174 255 255  66 196 255  18 200 182 255 255 255 

# Exercise 04

In [50]:

%%writefile ex4_volume.cu
#include <cstdio>
#include <cstdlib>
#include <sys/time.h>

#define NX 500
#define NY 500
#define NZ 500

#define CHECK(call)                                                   \
    do {                                                              \
        const cudaError_t error = (call);                            \
        if (error != cudaSuccess) {                                  \
            printf("Error: %s:%d, ", __FILE__, __LINE__);            \
            printf("code:%d, reason: %s\n", error,                   \
                   cudaGetErrorString(error));                       \
            exit(1);                                                 \
        }                                                             \
    } while (0)

double cpuSecond() {
    struct timeval tp;
    gettimeofday(&tp, NULL);
    return (double)tp.tv_sec + (double)tp.tv_usec * 1.e-6;
}

void random_ints(int *x, size_t size) {
    for (size_t i = 0; i < size; i++) {
        x[i] = rand() % 100;
    }
}

// Element-wise multiplication of two 3D matrices
__global__ void mulElem3D(int *a, int *b, int *c,
                          int nx, int ny, int nz) {

    // TODO 1: Calculate x, y and z coordinates
    int x = blockIdx.x * blockDim.x + threadIdx.x;
    int y = blockIdx.y * blockDim.y + threadIdx.y;
    int z = blockIdx.z * blockDim.z + threadIdx.z;

    // TODO 2: Check all three dimensions
    if (x < nx && y < ny && z < nz) {

        // TODO 3: Calculate flat array index
        size_t idx = ((size_t)z * ny + y) * nx + x;

        c[idx] = a[idx] * b[idx];
    }
}

int main(void) {

    size_t count = (size_t)NX * NY * NZ;
    size_t size = count * sizeof(int);

    int *a, *b, *c;
    int *d_a, *d_b, *d_c;

    double dStart, dElaps;

    // Allocate GPU memory
    CHECK(cudaMalloc((void **)&d_a, size));
    CHECK(cudaMalloc((void **)&d_b, size));
    CHECK(cudaMalloc((void **)&d_c, size));

    // Allocate CPU memory
    a = (int *)malloc(size);
    random_ints(a, count);

    b = (int *)malloc(size);
    random_ints(b, count);

    c = (int *)malloc(size);

    // Copy input arrays from CPU to GPU
    CHECK(cudaMemcpy(d_a, a, size, cudaMemcpyHostToDevice));
    CHECK(cudaMemcpy(d_b, b, size, cudaMemcpyHostToDevice));

    // 8 x 8 x 8 = 512 threads per block
    dim3 threadsPerBlock(8, 8, 8);

    // TODO 4: Calculate grid dimensions, rounding up
    dim3 numBlocks(
        (NX + threadsPerBlock.x - 1) / threadsPerBlock.x,
        (NY + threadsPerBlock.y - 1) / threadsPerBlock.y,
        (NZ + threadsPerBlock.z - 1) / threadsPerBlock.z
    );

    printf("Grid: %u x %u x %u blocks\n",
           numBlocks.x, numBlocks.y, numBlocks.z);

    // Start timing
    dStart = cpuSecond();

    // TODO 5: Launch CUDA kernel
    mulElem3D<<<numBlocks, threadsPerBlock>>>(
        d_a, d_b, d_c, NX, NY, NZ
    );

    CHECK(cudaGetLastError());
    CHECK(cudaDeviceSynchronize());

    dElaps = cpuSecond() - dStart;

    // Copy results back to CPU
    CHECK(cudaMemcpy(c, d_c, size, cudaMemcpyDeviceToHost));

    // TODO 6: Print the last 1000 elements
    for (size_t i = count - 1000; i < count; i++) {

        int x = i % NX;
        int y = (i / NX) % NY;
        int z = i / ((size_t)NX * NY);

        printf("C[%d][%d][%d] = %d x %d = %d\n",
               z, y, x, a[i], b[i], c[i]);
    }

    // TODO 7: Verify all results on CPU
    size_t errors = 0;

    for (size_t i = 0; i < count; i++) {
        if (c[i] != a[i] * b[i]) {
            errors++;
        }
    }

    printf("Verification: %zu mismatches out of %zu\n",
           errors, count);

    printf("Time taken - %.7f seconds\n", dElaps);

    // Cleanup
    free(a);
    free(b);
    free(c);

    CHECK(cudaFree(d_a));
    CHECK(cudaFree(d_b));
    CHECK(cudaFree(d_c));

    return 0;
}

Writing ex4_volume.cu


In [51]:
!nvcc -arch=sm_75 ex4_volume.cu -o ex4_volume && ./ex4_volume

Grid: 63 x 63 x 63 blocks
C[499][498][0] = 76 x 15 = 1140
C[499][498][1] = 38 x 93 = 3534
C[499][498][2] = 34 x 11 = 374
C[499][498][3] = 85 x 47 = 3995
C[499][498][4] = 3 x 38 = 114
C[499][498][5] = 43 x 82 = 3526
C[499][498][6] = 16 x 52 = 832
C[499][498][7] = 28 x 23 = 644
C[499][498][8] = 32 x 28 = 896
C[499][498][9] = 80 x 98 = 7840
C[499][498][10] = 69 x 81 = 5589
C[499][498][11] = 45 x 79 = 3555
C[499][498][12] = 30 x 2 = 60
C[499][498][13] = 41 x 13 = 533
C[499][498][14] = 30 x 81 = 2430
C[499][498][15] = 30 x 74 = 2220
C[499][498][16] = 41 x 13 = 533
C[499][498][17] = 14 x 18 = 252
C[499][498][18] = 83 x 20 = 1660
C[499][498][19] = 13 x 28 = 364
C[499][498][20] = 86 x 17 = 1462
C[499][498][21] = 13 x 6 = 78
C[499][498][22] = 88 x 57 = 5016
C[499][498][23] = 71 x 29 = 2059
C[499][498][24] = 28 x 17 = 476
C[499][498][25] = 78 x 2 = 156
C[499][498][26] = 50 x 37 = 1850
C[499][498][27] = 86 x 40 = 3440
C[499][498][28] = 28 x 35 = 980
C[499][498][29] = 15 x 1 = 15
C[499][498][30] =